In [1]:
import pandas as pd 
test = pd.read_csv('ship/test.csv')

In [2]:
numerical_test = test.select_dtypes(include=['int64', 'float64']).columns
categorical_test = test.select_dtypes(include='object').columns 

def missing(df, feature):
    total = df[feature].isnull().sum()
    percentage = (total / len(df)) * 100
    stats = pd.DataFrame({'missing': total, 'Percent': percentage})
    stats = stats[stats['missing'] > 0]
    return stats 

num = missing(test, numerical_test)
print('for numerical: ')
print(num)

cat = missing(test, categorical_test)
print(' \n for categorical: ')
print(cat)

print(len(numerical_test))
print(len(categorical_test))

print(numerical_test.tolist())
print(categorical_test.tolist())

for numerical: 
      missing    Percent
Age        86  20.574163
Fare        1   0.239234
 
 for categorical: 
       missing    Percent
Cabin      327  78.229665
6
5
['PassengerId', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']


In [3]:
import numpy as np

test['Age'] = test['Age'].fillna(test['Age'].median())

test['Fare'] = test['Fare'].fillna(test['Fare'].median())

test['Family'] = test['SibSp'] + test['Parch'] + 1
test['Fare_per_person'] = test['Fare'] / test['Family']
test['Fare_log'] = np.log1p(test['Fare_per_person'])


test_drop = ['SibSp', 'Parch', 'Fare', 'Name', 'Ticket', 'Cabin', 'Fare_per_person']
test = test.drop(columns= test_drop, errors = 'ignore')

In [4]:
from sklearn.preprocessing import LabelEncoder


numerical_test = test.select_dtypes(include=['int64', 'float64']).columns
categorical_test = test.select_dtypes(include='object').columns 

print(numerical_test.tolist())
print(categorical_test.tolist())

# Sex: LabelEncoder
le_sex = LabelEncoder()
test['Sex'] = le_sex.fit_transform(test['Sex'])  # male=0, female=1

# Family: LabelEncoder (Alone/NotAlone)
test['Family_label'] = test['Family'].apply(lambda x: 'Alone' if x == 1 else 'NotAlone')
le_family = LabelEncoder()
test['Family_label'] = le_family.fit_transform(test['Family_label'])  # Alone=0, NotAlone=1

# Pclass: OneHotEncoder (dùng pandas get_dummies cho tiện)
test = pd.get_dummies(test, columns=['Pclass'], prefix='Pclass')

# Embarked: OneHotEncoder
test = pd.get_dummies(test, columns=['Embarked'], prefix='Embarked')

['PassengerId', 'Pclass', 'Age', 'Family', 'Fare_log']
['Sex', 'Embarked']


In [6]:
import joblib
import pandas as pd

# Load model đã train
gb_model = joblib.load("ship/gradient_boosting_model.pkl")


# Predict
X_test = test.drop('PassengerId', axis=1)  # bỏ PassengerId khi predict
y_pred_test = gb_model.predict(X_test)

# Nếu muốn lưu kết quả ra file CSV
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": y_pred_test
})
submission.to_csv("ship/submission.csv", index=False)

print("Predict xong, file submission.csv đã sẵn sàng!")

Predict xong, file submission.csv đã sẵn sàng!
